In [ ]:
!pip install -q kagglehub tensorflow nltk scikit-learn pandas


In [ ]:


import kagglehub
import os

# Download the dataset again
path = kagglehub.dataset_download(
    "yash9439/eng-hinglish-machine-translation",
    force_download=True
)

print("Downloaded path:", path)
print("Files in folder:", os.listdir(path))

for root, dirs, files in os.walk(path):
    for file in files:
        print(os.path.join(root, file))


In [ ]:

import os
import pandas as pd

data_folder = os.path.join(path, "Eng - Hinglish")

train_file = os.path.join(data_folder, "train.txt")
test_file = os.path.join(data_folder, "test.txt")
dev_file = os.path.join(data_folder, "dev.txt")

# Display the first 10 lines of each file
for file_path in [train_file, test_file, dev_file]:
    print("\nFile:", os.path.basename(file_path))

    with open(file_path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            print(repr(line))
            if i == 9:
                break


In [ ]:

import os
import pandas as pd
import re

data_folder = os.path.join(path, "Eng - Hinglish")

def load_translation_file(filename):
    file_path = os.path.join(data_folder, filename)

    df = pd.read_csv(
        file_path,
        sep="\t",
        header=None,
        names=["english", "hinglish"],
        on_bad_lines="skip"
    )

    df = df.dropna()
    df["english"] = df["english"].astype(str).str.strip()
    df["hinglish"] = df["hinglish"].astype(str).str.strip()

    df = df[
        (df["english"] != "") &
        (df["hinglish"] != "")
    ]

    return df.reset_index(drop=True)

train_df = load_translation_file("train.txt")
test_df = load_translation_file("test.txt")
dev_df = load_translation_file("dev.txt")

print("Training pairs:", len(train_df))
print("Testing pairs:", len(test_df))
print("Validation pairs:", len(dev_df))

display(train_df.head())


In [ ]:

def clean_text(text):
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text

for frame in [train_df, test_df, dev_df]:
    frame["english"] = frame["english"].apply(clean_text)
    frame["hinglish"] = frame["hinglish"].apply(clean_text)

train_df = train_df.drop_duplicates().reset_index(drop=True)

print("Preprocessing completed!")
display(train_df.head())


In [ ]:

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

MAX_VOCAB = 15000

# Add start and end tokens to Hinglish sentences
train_hin = [
    "startseq " + s + " endseq" for s in train_df["hinglish"]
]
dev_hin = [
    "startseq " + s + " endseq" for s in dev_df["hinglish"]
]
test_hin = [
    "startseq " + s + " endseq" for s in test_df["hinglish"]
]

# Create tokenizers
eng_tokenizer = Tokenizer(
    num_words=MAX_VOCAB, oov_token="<unk>"
)
hin_tokenizer = Tokenizer(
    num_words=MAX_VOCAB, oov_token="<unk>", filters=""
)

eng_tokenizer.fit_on_texts(train_df["english"])
hin_tokenizer.fit_on_texts(train_hin)

# Convert sentences to sequences
X_train = eng_tokenizer.texts_to_sequences(train_df["english"])
X_dev = eng_tokenizer.texts_to_sequences(dev_df["english"])
X_test = eng_tokenizer.texts_to_sequences(test_df["english"])

Y_train = hin_tokenizer.texts_to_sequences(train_hin)
Y_dev = hin_tokenizer.texts_to_sequences(dev_hin)
Y_test = hin_tokenizer.texts_to_sequences(test_hin)

# Set maximum sequence lengths
MAX_ENG_LEN = min(
    40, max(len(s) for s in X_train)
)
MAX_HIN_LEN = min(
    45, max(len(s) for s in Y_train)
)

# Pad sequences
X_train = pad_sequences(X_train, maxlen=MAX_ENG_LEN, padding="post", truncating="post")
X_dev = pad_sequences(X_dev, maxlen=MAX_ENG_LEN, padding="post", truncating="post")
X_test = pad_sequences(X_test, maxlen=MAX_ENG_LEN, padding="post", truncating="post")

Y_train = pad_sequences(Y_train, maxlen=MAX_HIN_LEN, padding="post", truncating="post")
Y_dev = pad_sequences(Y_dev, maxlen=MAX_HIN_LEN, padding="post", truncating="post")
Y_test = pad_sequences(Y_test, maxlen=MAX_HIN_LEN, padding="post", truncating="post")

# Teacher forcing: previous tokens are inputs, next tokens are targets
decoder_input_train = Y_train[:, :-1]
decoder_target_train = Y_train[:, 1:]

decoder_input_dev = Y_dev[:, :-1]
decoder_target_dev = Y_dev[:, 1:]

decoder_input_test = Y_test[:, :-1]
decoder_target_test = Y_test[:, 1:]

# Ignore padding tokens during loss calculation
train_weights = (decoder_target_train != 0).astype("float32")
dev_weights = (decoder_target_dev != 0).astype("float32")
test_weights = (decoder_target_test != 0).astype("float32")

eng_vocab_size = min(
    MAX_VOCAB, len(eng_tokenizer.word_index) + 1
)
hin_vocab_size = min(
    MAX_VOCAB, len(hin_tokenizer.word_index) + 1
)

print("English vocabulary:", eng_vocab_size)
print("Hinglish vocabulary:", hin_vocab_size)
print("Training shape:", X_train.shape)
print("Decoder target shape:", decoder_target_train.shape)


In [ ]:

import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense
from tensorflow.keras.models import Model

EMBEDDING_DIM = 128
LSTM_UNITS = 256

# Encoder
encoder_inputs = Input(shape=(MAX_ENG_LEN,), name="encoder_inputs")

encoder_embedding_layer = Embedding(
    eng_vocab_size, EMBEDDING_DIM, mask_zero=True
)
encoder_embedding = encoder_embedding_layer(encoder_inputs)

encoder_lstm = LSTM(
    LSTM_UNITS, return_state=True, name="encoder_lstm"
)

_, state_h, state_c = encoder_lstm(encoder_embedding)
encoder_states = [state_h, state_c]

# Decoder
decoder_inputs = Input(
    shape=(MAX_HIN_LEN - 1,), name="decoder_inputs"
)

decoder_embedding_layer = Embedding(
    hin_vocab_size, EMBEDDING_DIM, mask_zero=True
)
decoder_embedding = decoder_embedding_layer(decoder_inputs)

decoder_lstm = LSTM(
    LSTM_UNITS, return_sequences=True,
    return_state=True, name="decoder_lstm"
)

decoder_outputs, _, _ = decoder_lstm(
    decoder_embedding, initial_state=encoder_states
)

decoder_dense = Dense(hin_vocab_size, activation="softmax")
decoder_outputs = decoder_dense(decoder_outputs)

model = Model(
    [encoder_inputs, decoder_inputs], decoder_outputs
)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy"
)

model.summary()


In [ ]:

from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

history = model.fit(
    [X_train, decoder_input_train],
    decoder_target_train,
    sample_weight=train_weights,
    validation_data=(
        [X_dev, decoder_input_dev],
        decoder_target_dev,
        dev_weights
    ),
    epochs=10,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)


Epoch 1/10
104/104 ━━━━━━━━━━━━━━━━━━━━ 267s 3s/step - loss: 6.2309 - val_loss: 5.6878
Epoch 2/10
104/104 ━━━━━━━━━━━━━━━━━━━━ 281s 3s/step - loss: 5.5145 - val_loss: 5.5333
Epoch 3/10
104/104 ━━━━━━━━━━━━━━━━━━━━ 267s 3s/step - loss: 5.3034 - val_loss: 5.3963
Epoch 4/10
 95/104 ━━━━━━━━━━━━━━━━━━━━ 22s 2s/step - loss: 5.1009

In [ ]:

import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("LSTM Model Training")
plt.legend()
plt.show()


In [ ]:

# Encoder inference model
encoder_model = Model(encoder_inputs, encoder_states)

# Decoder inference inputs
state_input_h = Input(shape=(LSTM_UNITS,))
state_input_c = Input(shape=(LSTM_UNITS,))
token_input = Input(shape=(1,))

token_embedding = decoder_embedding_layer(token_input)

step_output, next_h, next_c = decoder_lstm(
    token_embedding,
    initial_state=[state_input_h, state_input_c]
)

step_output = decoder_dense(step_output)

decoder_model = Model(
    [token_input, state_input_h, state_input_c],
    [step_output, next_h, next_c]
)

word_index = hin_tokenizer.word_index
index_word = hin_tokenizer.index_word

start_id = word_index["startseq"]
end_id = word_index["endseq"]

def translate(sentence):
    sequence = eng_tokenizer.texts_to_sequences([clean_text(sentence)])
    sequence = pad_sequences(
        sequence,
        maxlen=MAX_ENG_LEN,
        padding="post",
        truncating="post"
    )

    states = encoder_model.predict(sequence, verbose=0)
    token = np.array([[start_id]])
    words = []

    for _ in range(MAX_HIN_LEN - 1):
        output, h, c = decoder_model.predict(
            [token] + states, verbose=0
        )

        predicted_id = int(np.argmax(output[0, -1, :]))

        if predicted_id == end_id or predicted_id == 0:
            break

        word = index_word.get(predicted_id, "<unk>")

        if word != "startseq":
            words.append(word)

        token = np.array([[predicted_id]])
        states = [h, c]

    return " ".join(words)

print(translate("How are you?"))


In [ ]:

unseen_sentences = [
    "How are you?",
    "What is your name?",
    "I am going to school.",
    "Please help me.",
    "I will call you tomorrow."
]

for sentence in unseen_sentences:
    print("English:", sentence)
    print("Hinglish:", translate(sentence))
    print("-" * 40)


In [ ]:

from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

references = []
hypotheses = []

n = min(200, len(test_df))
smooth = SmoothingFunction().method1

for i in range(n):
    actual = test_df.iloc[i]["hinglish"]
    predicted = translate(test_df.iloc[i]["english"])

    references.append([actual.split()])
    hypotheses.append(predicted.split())

bleu = corpus_bleu(
    references,
    hypotheses,
    smoothing_function=smooth
)

print("BLEU Score:", round(bleu, 4))
print("BLEU Score (%):", round(bleu * 100, 2))
